<a href="https://colab.research.google.com/github/manaf2808/Finance/blob/main/IntBusFin_InterestParityipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from google.colab import files

In [2]:
spot_forwards = pd.read_excel("Spot_Forwards_USDEUR.xlsx")
spot_forwards.head()

,Dates,Spot_USDEUR,USDEUR1W,USDEUR1M,USDEUR6M,USDEUR12M,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,2000-01-03,0.9763,NaN,NaN,NaN,NaN,NaN,-16.27077,-32.82124,-154.57093,-299.10962
1,2000-01-04,0.9713,NaN,NaN,NaN,NaN,NaN,8.05000,14.31662,42.08117,282.14994
2,2000-01-05,0.9689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,0.9682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,0.9713,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
deposit = pd.read_excel("Deposit_USDEUR.xlsx")
deposit.head()

,Dates,USD_1W,USD_1M,USD_6M,USD_12M,EUR_1W,EUR_1M,EUR_6M,EUR_12M
0,2000-01-03,5.75,5.78,6.18,6.51,3.050,3.080,3.430,3.830
1,2000-01-04,5.60,5.78,6.21,6.62,3.025,3.100,3.490,3.900
2,2000-01-05,5.57,5.78,6.20,6.55,2.985,3.135,3.535,3.935
3,2000-01-06,5.57,5.75,6.20,6.58,2.970,3.060,3.470,3.880
4,2000-01-07,5.57,5.76,6.20,6.58,2.985,3.115,3.525,3.935


In [4]:
spot_forwards['Dates'] = pd.to_datetime(spot_forwards['Dates'])
deposit['Dates'] = pd.to_datetime(deposit['Dates'])

df = pd.merge(spot_forwards, deposit, on='Dates', how='inner')

df.head()

,Dates,Spot_USDEUR,USDEUR1W,USDEUR1M,USDEUR6M,USDEUR12M,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,USD_1W,USD_1M,USD_6M,USD_12M,EUR_1W,EUR_1M,EUR_6M,EUR_12M
0,2000-01-03,0.9763,NaN,NaN,NaN,NaN,NaN,-16.27077,-32.82124,-154.57093,-299.10962,5.75,5.78,6.18,6.51,3.050,3.080,3.430,3.830
1,2000-01-04,0.9713,NaN,NaN,NaN,NaN,NaN,8.05000,14.31662,42.08117,282.14994,5.60,5.78,6.21,6.62,3.025,3.100,3.490,3.900
2,2000-01-05,0.9689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.57,5.78,6.20,6.55,2.985,3.135,3.535,3.935
3,2000-01-06,0.9682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.57,5.75,6.20,6.58,2.970,3.060,3.470,3.880
4,2000-01-07,0.9713,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.57,5.76,6.20,6.58,2.985,3.115,3.525,3.935


In [5]:
start_date = '2008-10-03'  #This date is where all the forwards are available aways to skip the data cleaning

df = df[df['Dates'] >= start_date].copy()
df.reset_index(drop=True, inplace=True)

df.head()

,Dates,Spot_USDEUR,USDEUR1W,USDEUR1M,USDEUR6M,USDEUR12M,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,USD_1W,USD_1M,USD_6M,USD_12M,EUR_1W,EUR_1M,EUR_6M,EUR_12M
0,2008-10-03,0.7260,4.25,-4.97253,-27.20000,167.85003,NaN,NaN,NaN,NaN,NaN,4.25,5.125,4.96,4.36,4.75,5.100,5.380,5.490
1,2008-10-06,0.7408,4.25,-2.44092,-25.89999,167.85003,NaN,NaN,NaN,NaN,NaN,4.25,4.875,4.60,4.37,4.75,5.125,5.400,5.475
2,2008-10-07,0.7359,4.25,-6.12526,-10.50000,167.85003,NaN,NaN,NaN,NaN,NaN,4.25,5.000,4.82,4.58,5.00,5.125,5.500,5.510
3,2008-10-08,0.7324,4.25,-19.80877,13.50000,167.85003,NaN,NaN,NaN,NaN,NaN,4.50,4.750,5.50,5.11,5.00,5.175,5.500,5.435
4,2008-10-09,0.7351,4.25,-13.72014,22.50000,167.85003,NaN,NaN,NaN,NaN,NaN,5.00,5.500,5.77,5.22,4.35,5.125,5.375,5.475


In [6]:
column_names = df.columns.tolist()
print(column_names)

['Dates', 'Spot_USDEUR', 'USDEUR1W', 'USDEUR1M', 'USDEUR6M', 'USDEUR12M', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'USD_1W', 'USD_1M', 'USD_6M', 'USD_12M', 'EUR_1W', 'EUR_1M', 'EUR_6M', 'EUR_12M']


In [7]:
df['FWD_1W'] = df['Spot_USDEUR'] + df['USDEUR1W'] / 10000
df['FWD_1M'] = df['Spot_USDEUR'] + df['USDEUR1M'] / 10000
df['FWD_6M'] = df['Spot_USDEUR'] + df['USDEUR6M'] / 10000
df['FWD_12M'] = df['Spot_USDEUR'] + df['USDEUR12M'] / 10000

df['USDEUR1W'] = df['FWD_1W']
df['USDEUR1M'] = df['FWD_1M']
df['USDEUR6M'] = df['FWD_6M']
df['USDEUR12M'] = df['FWD_12M']

df.drop(columns=['FWD_1W','FWD_1M','FWD_6M','FWD_12M'], inplace=True)


In [8]:
horizons = ['1W', '1M', '6M', '12M']

forward_cols = {
    '1W': 'USDEUR1W',
    '1M': 'USDEUR1M',
    '6M': 'USDEUR6M',
    '12M': 'USDEUR12M'
}

us_ir_cols = {
    '1W': 'USD_1W',
    '1M': 'USD_1M',
    '6M': 'USD_6M',
    '12M': 'USD_12M'
}

eur_ir_cols = {
    '1W': 'EUR_1W',
    '1M': 'EUR_1M',
    '6M': 'EUR_6M',
    '12M': 'EUR_12M'
}

In [9]:
for h in horizons:
    f_col = forward_cols[h]
    usd_col = us_ir_cols[h]
    eur_col = eur_ir_cols[h]

    fig = go.Figure()

    # Spot rate
    fig.add_trace(go.Scatter(
        x=df['Dates'], y=df['Spot_USDEUR'],
        mode='lines', name='Spot rate (USDEUR)'
    ))

    # Forward rate
    fig.add_trace(go.Scatter(
        x=df['Dates'], y=df[f_col],
        mode='lines', name=f'Forward {h}'
    ))

    # Foreign interest rate (USD)
    fig.add_trace(go.Scatter(
        x=df['Dates'], y=df[usd_col],
        mode='lines', name=f'USD rate {h}',
        yaxis='y2'
    ))

    # Domestic interest rate (EUR)
    fig.add_trace(go.Scatter(
        x=df['Dates'], y=df[eur_col],
        mode='lines', name=f'EUR rate {h}',
        yaxis='y2'
    ))

    # Layout with dual y-axis (FX on left, rates on right)
    fig.update_layout(
        title=f'USDEUR – Spot, Forward and Interest Rates ({h})',
        xaxis_title='Date',
        yaxis=dict(title='Exchange rate (USDEUR)'),
        yaxis2=dict(
            title='Interest rates',
            overlaying='y',
            side='right'
        ),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )

    fig.show()

In [10]:
desc_tables = {}

for h in horizons:
    f_col = forward_cols[h]
    usd_col = us_ir_cols[h]
    eur_col = eur_ir_cols[h]

    subset = df[['Spot_USDEUR', f_col, usd_col, eur_col]].copy()
    desc = subset.describe().T
    desc = desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
    desc = desc.round(6)

    desc_tables[h] = desc
    print(f"\n================================ Descriptive Statistics for {h} ==============================")
    print(desc.to_markdown())


================================ Descriptive Statistics for 1W ==============================
|             |     mean |      std |       min |       25% |      50% |      75% |     max |
|:------------|---------:|---------:|----------:|----------:|---------:|---------:|--------:|
| Spot_USDEUR | 0.841265 | 0.081341 |  0.6608   |  0.7656   | 0.8572   | 0.9077   | 1.0423  |
| USDEUR1W    | 0.84113  | 0.081147 |  0.661225 |  0.765594 | 0.856818 | 0.907502 | 1.04174 |
| USD_1W      | 1.41673  | 1.76711  |  0        |  0.175    | 0.345    | 2.235    | 5.6225  |
| EUR_1W      | 0.564549 | 1.34254  | -0.65     | -0.42     | 0.07     | 0.82     | 5       |

================================ Descriptive Statistics for 1M ==============================
|             |     mean |      std |       min |      25% |      50% |      75% |     max |
|:------------|---------:|---------:|----------:|---------:|---------:|---------:|--------:|
| Spot_USDEUR | 0.841265 | 0.081341 |  0.6608   |  0.7656  |

In [11]:
for h in horizons:
    f_col = forward_cols[h]
    usd_col = us_ir_cols[h]
    eur_col = eur_ir_cols[h]

    df[f'cip_dep_{h}'] = np.log(df[f_col]) - np.log(df['Spot_USDEUR'])
    df[f'cip_exp_{h}'] = df[eur_col] - df[usd_col]

In [12]:
results = []

for h in horizons:
    y = df[f'cip_dep_{h}']
    x = df[f'cip_exp_{h}']

    reg_df = pd.DataFrame({'y': y, 'x': x}).dropna()

    X = sm.add_constant(reg_df['x'])
    model = sm.OLS(reg_df['y'], X).fit()

    results.append({
        'Horizon': h,
        'alpha': model.params['const'],
        'beta': model.params['x'],
        'alpha_pval': model.pvalues['const'],
        'beta_pval': model.pvalues['x'],
        'R2': model.rsquared,
        'N': int(model.nobs)
    })

    x_vals = reg_df['x']
    y_vals = reg_df['y']

    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_line = model.params['const'] + model.params['x'] * x_line

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='markers',
        name='Data points',
        marker=dict(size=4, opacity=0.5)
    ))

    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        name='Regression line',
        line=dict(color='red', width=2)
    ))

    fig.update_layout(
        title=f'CIP Regression Scatter – {h}',
        xaxis_title='Interest Rate Differential (USD - EUR)',
        yaxis_title='log(F) - log(S)'
    )

    fig.show()

In [13]:
reg_summary = pd.DataFrame(results)
reg_summary = reg_summary.round(6)
print(reg_summary.to_markdown(index=False))

| Horizon   |     alpha |     beta |   alpha_pval |   beta_pval |       R2 |    N |
|:----------|----------:|---------:|-------------:|------------:|---------:|-----:|
| 1W        |  8e-05    | 0.000256 |            0 |           0 | 0.712221 | 4499 |
| 1M        | -0.000187 | 0.000818 |            0 |           0 | 0.94018  | 4499 |
| 6M        | -0.001129 | 0.004484 |            0 |           0 | 0.909342 | 4499 |
| 12M       |  0.002733 | 0.011064 |            0 |           0 | 0.757721 | 4499 |
